# Movie Recommender Data Preparation

This notebook cleans the MovieLens-style datasets, merges movie metadata with ratings, and engineers features for recommendation modeling.

## 1. Imports and setup

We keep the imports minimal and use consistent naming across all tables.

In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
#import tensorflow as tf

pd.set_option('display.max_colwidth', 120)
np.random.seed(42)

## 2. Load raw tables

The raw data comes from credits, keywords, movies metadata, and ratings.

In [2]:
credits_df = pd.read_csv('ml-dataset/credits.csv')
keywords_df = pd.read_csv('ml-dataset/keywords.csv')
movies_df = pd.read_csv('ml-dataset/movies_metadata.csv', low_memory=False)
ratings_df = pd.read_csv('ml-dataset/ratings.csv')

display(credits_df.head())
display(keywords_df.head())
display(movies_df.head())
display(ratings_df.head())

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name...","[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'n...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', 'credit_id': '52fe44bfc3a36847f80a7c73', 'gender': 2, 'id': 2157, 'name...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'department': 'Production', 'gender': 2, 'id': 511, 'job': 'Executive Pro...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'credit_id': '52fe466a9251416c75077a8d', 'gender': 2, 'id': 6837, 'name'...","[{'credit_id': '52fe466a9251416c75077a89', 'department': 'Directing', 'gender': 2, 'id': 26502, 'job': 'Director', '...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah' Jackson"", 'credit_id': '52fe44779251416c91011aad', 'gender': 1, 'id'...","[{'credit_id': '52fe44779251416c91011acb', 'department': 'Directing', 'gender': 2, 'id': 2178, 'job': 'Director', 'n...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', 'credit_id': '52fe44959251416c75039eb9', 'gender': 2, 'id': 67773, 'nam...","[{'credit_id': '52fe44959251416c75039ed7', 'department': 'Sound', 'gender': 2, 'id': 37, 'job': 'Original Music Comp...",11862


,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'fr..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 10941, 'name': 'disappearance'}, {'id': 15101, 'name': ""based on childr..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392, 'name': 'best friend'}, {'id': 179431, 'name': 'duringcreditsstinger..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id': 10131, 'name': 'interracial relationship'}, {'id': 14768, 'name': 'si..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'name': 'midlife crisis'}, {'id': 2246, 'name': 'confidence'}, {'id': 49..."


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqgPgm3F85NMCii9gVFUcet.jpg', 'backdrop_pat...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",NaN,15602,tt0113228,en,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile,...",...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to bre...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself... and never let you forget it.,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Collection', 'poster_path': '/nts4iOmNnq7GNicycMJ9pSAn204.jpg', 'backdrop...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,"Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and t...",...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's In For The Surprise Of His Life!,Father of the Bride Part II,False,5.7,173.0


,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


## 3. Clean credits and keywords

We parse list-like strings safely and keep only the fields needed for recommendation features.

In [3]:
def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return x if isinstance(x, list) else []


def extract_top_cast(cast_list, n=5):
    if not isinstance(cast_list, list):
        return []
    return [item.get('name') for item in cast_list[:n] if isinstance(item, dict) and item.get('name')]


def extract_director(crew_list):
    if not isinstance(crew_list, list):
        return None
    for person in crew_list:
        if isinstance(person, dict) and person.get('job') == 'Director':
            return person.get('name')
    return None


def extract_writers(crew_list):
    if not isinstance(crew_list, list):
        return []
    roles = {'Writer', 'Screenplay', 'Story'}
    writers = []
    for person in crew_list:
        if isinstance(person, dict) and person.get('job') in roles and person.get('name'):
            writers.append(person['name'])
    return writers

credits_df['cast'] = credits_df['cast'].apply(safe_literal_eval)
credits_df['crew'] = credits_df['crew'].apply(safe_literal_eval)
credits_df['top_cast'] = credits_df['cast'].apply(extract_top_cast)
credits_df['director'] = credits_df['crew'].apply(extract_director)
credits_df['writers'] = credits_df['crew'].apply(extract_writers)
credits_df = credits_df.drop(columns=['cast', 'crew'])

keywords_df['keywords'] = keywords_df['keywords'].apply(safe_literal_eval)
keywords_df['keyword_names'] = keywords_df['keywords'].apply(lambda items: [item.get('name') for item in items if isinstance(item, dict) and item.get('name')])
keywords_df = keywords_df.drop(columns=['keywords'])

display(credits_df.head())
display(keywords_df.head())

,id,top_cast,director,writers
0,862,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn]",John Lasseter,"[Joss Whedon, Andrew Stanton, Joel Cohen, Alec Sokolow]"
1,8844,"[Robin Williams, Jonathan Hyde, Kirsten Dunst, Bradley Pierce, Bonnie Hunt]",Joe Johnston,"[Jonathan Hensleigh, Greg Taylor, Jim Strain]"
2,15602,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sophia Loren, Daryl Hannah]",Howard Deutch,[Mark Steven Johnson]
3,31357,"[Whitney Houston, Angela Bassett, Loretta Devine, Lela Rochon, Gregory Hines]",Forest Whitaker,"[Ronald Bass, Terry McMillan]"
4,11862,"[Steve Martin, Diane Keaton, Martin Short, Kimberly Williams-Paisley, George Newbern]",Charles Shyer,"[Nancy Meyers, Albert Hackett]"


,id,keyword_names
0,862,"[jealousy, toy, boy, friendship, friends, rivalry, boy next door, new toy, toy comes to life]"
1,8844,"[board game, disappearance, based on children's book, new home, recluse, giant insect]"
2,15602,"[fishing, best friend, duringcreditsstinger, old men]"
3,31357,"[based on novel, interracial relationship, single mother, divorce, chick flick]"
4,11862,"[baby, midlife crisis, confidence, aging, daughter, mother daughter relationship, pregnancy, contraception, gynecolo..."


## 4. Clean movie metadata

We standardize list columns, parse collection information, and drop duplicate movie IDs.

In [4]:
movies_df = movies_df[['id', 'original_title', 'overview', 'tagline', 'genres', 'production_companies',
                       'production_countries', 'spoken_languages', 'release_date', 'runtime', 'popularity',
                       'vote_average', 'vote_count', 'belongs_to_collection']].copy()


def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    if isinstance(x, list):
        return x
    return []


def extract_names(x):
    if not isinstance(x, list):
        return []
    return [item.get('name') for item in x if isinstance(item, dict) and item.get('name')]


def extract_collection_name(x):
    if pd.isna(x):
        return None
    if isinstance(x, dict):
        return x.get('name')
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, dict):
                return parsed.get('name')
        except (ValueError, SyntaxError):
            return None
    return None


list_cols = ['genres', 'production_companies', 'production_countries', 'spoken_languages']
for col in list_cols:
    movies_df[col] = movies_df[col].apply(safe_literal_eval).apply(extract_names)

movies_df['belongs_to_collection'] = movies_df['belongs_to_collection'].apply(extract_collection_name)
movies_df['id'] = pd.to_numeric(movies_df['id'], errors='coerce')
movies_df['popularity'] = pd.to_numeric(movies_df['popularity'], errors='coerce')
movies_df['vote_count'] = pd.to_numeric(movies_df['vote_count'], errors='coerce')
movies_df['vote_average'] = pd.to_numeric(movies_df['vote_average'], errors='coerce')
movies_df['runtime'] = pd.to_numeric(movies_df['runtime'], errors='coerce')

movies_df = movies_df.dropna(subset=['id'])
movies_df['id'] = movies_df['id'].astype(int)
movies_df = movies_df.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)

display(movies_df.head())

,id,original_title,overview,tagline,genres,production_companies,production_countries,spoken_languages,release_date,runtime,popularity,vote_average,vote_count,belongs_to_collection
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afrai...",NaN,"[Animation, Comedy, Family]",[Pixar Animation Studios],[United States of America],[English],1995-10-30,81.0,21.946943,7.7,5415.0,Toy Story Collection
1,8844,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwitting...",Roll the dice and unleash the excitement!,"[Adventure, Fantasy, Family]","[TriStar Pictures, Teitler Film, Interscope Communications]",[United States of America],"[English, Français]",1995-12-15,104.0,17.015539,6.9,2413.0,NaN
2,15602,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile,...",Still Yelling. Still Fighting. Still Ready for Love.,"[Romance, Comedy]","[Warner Bros., Lancaster Gate]",[United States of America],[English],1995-12-22,101.0,11.712900,6.5,92.0,Grumpy Old Men Collection
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to bre...",Friends are the people who let you be yourself... and never let you forget it.,"[Comedy, Drama, Romance]",[Twentieth Century Fox Film Corporation],[United States of America],[English],1995-12-22,127.0,3.859495,6.1,34.0,NaN
4,11862,Father of the Bride Part II,"Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and t...",Just When His World Is Back To Normal... He's In For The Surprise Of His Life!,[Comedy],"[Sandollar Productions, Touchstone Pictures]",[United States of America],[English],1995-02-10,106.0,8.387519,5.7,173.0,Father of the Bride Collection


## 5. Clean ratings and merge

We keep only ratings that point to valid movie IDs and ensure the merged tables stay aligned.

In [5]:
# 1. Merge first
merged_df = credits_df.merge(keywords_df, on='id', how='inner') \
                      .merge(movies_df, on='id', how='inner')

# 2. Clean IDs
merged_df['id'] = pd.to_numeric(merged_df['id'], errors='coerce')

# 3. Filter ratings using merged IDs
valid_ids = set(merged_df['id'].dropna())
ratings_df = ratings_df[ratings_df['movieId'].isin(valid_ids)]

# 4. Apply ≥4 ratings filter
rating_counts = ratings_df.groupby('movieId').size()
valid_movie_ids = set(rating_counts[rating_counts >= 4].index)

ratings_df = ratings_df[ratings_df['movieId'].isin(valid_movie_ids)]
merged_df = merged_df[merged_df['id'].isin(valid_movie_ids)]

# 5. 🔥 Deduplicate AFTER merge (critical)
merged_df = (
    merged_df
    .assign(_non_null=merged_df.notna().sum(axis=1))
    .sort_values('_non_null', ascending=False)
    .drop_duplicates(subset='id', keep='first')
    .drop(columns='_non_null')
    .reset_index(drop=True)
)

# 6. Final checks
assert merged_df['id'].duplicated().sum() == 0
assert set(ratings_df['movieId']) == set(merged_df['id'])

In [28]:
merged_df['keyword_names']

0       [holiday, corruption, double life, dc comics, crime fighter, hallucination, christmas tree, gotham city, vigilante, ...
1       [flying car, street gang, running, graduation, musical, rivalry, based on play or musical, gossip, makeover, automob...
2                                                                                  [london england, education, teacher, school]
3                                                                  [cook, war ship, mercenary, nuclear missile, hostage-taking]
4       [mayor, island, police chief, sailing, boat accident, dying and death, panic, current, aggression by animal, sequel,...
                                                                 ...                                                           
6263                                                                                                                         []
6264                                                                                                    

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

kw_strings = merged_df['keyword_names'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
vec = CountVectorizer(max_features=50, binary=False)
kw_matrix = vec.fit_transform(kw_strings).toarray()
kw_cols = [f'kw_{k}' for k in vec.get_feature_names_out()]

In [27]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# df should have a column called keyword_names containing lists of keywords
# Example: ["holiday", "corruption", "double life", ...]

def clean_keywords(x):
    if isinstance(x, list):
        return [str(k).strip().lower() for k in x if isinstance(k, (str, int, float)) and str(k).strip()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        return [k.strip().lower() for k in x.split(",") if k.strip()]
    return []

df = merged_df.copy()
df["keyword_names"] = df["keyword_names"].apply(clean_keywords)

# Option 1: multi-hot / count features
tfidf = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    min_df=2,
    max_df=0.8
)

X_kw = tfidf.fit_transform(df["keyword_names"])

# Optional: reduce keyword matrix size
svd = TruncatedSVD(n_components=50, random_state=42)
X_kw_reduced = svd.fit_transform(X_kw)

# Optional: normalize
X_kw_reduced = normalize(X_kw_reduced, norm="l2",axis=1)

# Add back to dataframe if needed
kw_cols = [f"kw_svd_{i}" for i in range(X_kw_reduced.shape[1])]
kw_df = pd.DataFrame(X_kw_reduced, columns=kw_cols, index=df.index)

kw_df

,kw_svd_0,kw_svd_1,kw_svd_2,kw_svd_3,kw_svd_4,kw_svd_5,kw_svd_6,kw_svd_7,kw_svd_8,kw_svd_9,...,kw_svd_40,kw_svd_41,kw_svd_42,kw_svd_43,kw_svd_44,kw_svd_45,kw_svd_46,kw_svd_47,kw_svd_48,kw_svd_49
0,0.002576,0.011520,0.088197,0.170888,-0.012349,-0.096285,-0.014808,-0.042365,0.133390,0.042065,...,-0.031590,-0.089127,-0.180701,0.176179,-0.291585,-0.145544,-0.140695,0.039553,0.090175,-0.056487
1,0.002377,0.016461,0.879849,-0.249522,-0.010617,0.020013,-0.019752,-0.001853,-0.010116,0.000912,...,0.147610,-0.038890,0.109170,-0.008846,0.139357,0.142069,0.086251,-0.102447,0.002452,0.114209
2,0.006460,0.017794,0.054610,0.101568,0.010084,-0.021299,0.006687,-0.040583,0.096473,-0.011502,...,0.118569,0.145280,0.046752,-0.266010,0.115615,0.053576,0.127887,0.261064,0.171627,-0.078405
3,0.024312,0.007845,0.076225,0.201684,-0.006383,-0.135714,0.016455,-0.037281,0.141083,0.285251,...,-0.042465,-0.455673,-0.359885,0.083848,-0.083498,-0.077983,-0.040729,-0.158931,0.224862,0.264967
4,0.007510,0.017200,0.070049,0.180388,-0.001174,-0.138404,0.002766,-0.071059,0.169763,-0.027967,...,0.102143,-0.191344,-0.111479,0.027119,-0.047845,-0.033484,0.093144,0.034089,-0.053688,-0.020954
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6263,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6264,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6265,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6266,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 6. Feature engineering

We create user-level, item-level, and content-based features without changing the number of rating rows.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# -----------------------------
# 1. Copy and parse metadata
# -----------------------------
meta_df = merged_df.copy()

for col in ['genres', 'top_cast', 'writers']:
    meta_df[col] = meta_df[col].apply(safe_literal_eval)

# -----------------------------
# 2. Multi-label binarization (genres)
# -----------------------------
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(meta_df['genres'])

genre_cols = [f'genre_{g}' for g in mlb.classes_]

genre_df = pd.DataFrame(
    genre_matrix,
    columns=genre_cols,
    index=meta_df.index
)

meta_df = pd.concat([meta_df, genre_df], axis=1)

# -----------------------------
# 3. Feature engineering (movie metadata)
# -----------------------------
meta_df['log_popularity'] = np.log1p(meta_df['popularity'])
meta_df['log_vote_count'] = np.log1p(meta_df['vote_count'])

meta_df['release_year'] = pd.to_datetime(
    meta_df['release_date'], errors='coerce'
).dt.year

meta_df['runtime'] = meta_df['runtime'].fillna(meta_df['runtime'].median())

meta_df['in_collection'] = meta_df['belongs_to_collection'].notna().astype(int)

meta_df['collection_size'] = (
    meta_df.groupby('belongs_to_collection')['id']
    .transform('count')
    .fillna(0)
    .astype(int)
)

meta_df['collection_id'] = (
    meta_df['belongs_to_collection']
    .astype('category')
    .cat.codes
    .where(meta_df['belongs_to_collection'].notna(), other=0)
)

# -----------------------------
# 4. Core ratings table
# -----------------------------
#ratings_core = ratings_df[['userId', 'movieId', 'rating', 'timestamp']].copy()
# Only keep positive interactions
ratings_core = ratings_df[ratings_df['rating'] >= 4.0][['userId', 'movieId', 'rating', 'timestamp']].copy()

# -----------------------------
# 5. User-level stats
# -----------------------------
user_stats = ratings_core.groupby('userId')['rating'].agg(
    user_mean_rating='mean',
    user_rating_std='std',
    user_rating_count='count'
).reset_index().fillna(0)

# -----------------------------
# 6. Movie-level stats
# -----------------------------
movie_stats = ratings_core.groupby('movieId')['rating'].agg(
    cf_mean_rating='mean',
    cf_rating_std='std',
    cf_rating_count='count'
).reset_index().fillna(0)

movie_stats['log_cf_count'] = np.log1p(movie_stats['cf_rating_count'])

# -----------------------------
# 7. User-centered rating deviation (optional but useful)
# -----------------------------
user_mean_map = user_stats.set_index('userId')['user_mean_rating']

ratings_core['rating_deviation'] = (
    ratings_core['rating'] - ratings_core['userId'].map(user_mean_map)
)

# -----------------------------
# 8. Merge ratings with genre features
# -----------------------------
ratings_with_genres = ratings_core.merge(
    meta_df[['id'] + genre_cols],
    left_on='movieId',
    right_on='id',
    how='left'
).drop(columns='id')

# -----------------------------
# 9. TRUE affinity (rating-weighted)
# -----------------------------
ratings_with_genres[genre_cols] = ratings_with_genres[genre_cols].mul(
    ratings_with_genres['rating'],
    axis=0
)

#affinity_df = ratings_with_genres.groupby('userId')[genre_cols].mean().reset_index()

#affinity_df = affinity_df.rename(
#    columns=lambda c: c.replace('genre_', 'affinity_') if c.startswith('genre_') else c
#)

# -----------------------------
# 10. Final feature table
# -----------------------------
# final_df = (
#     ratings_core
#     .merge(user_stats, on='userId', how='left')
#     .merge(movie_stats, on='movieId', how='left')
#     .merge(affinity_df, on='userId', how='left')
#     .merge(
#         meta_df[['id'] + genre_cols + [
#             'log_popularity',
#             'log_vote_count',
#             'release_year',
#             'runtime',
#             'in_collection',
#             'collection_size',
#             'collection_id',
#             "vote_average",
#             "overview"
#         ]],
#         left_on='movieId',
#         right_on='id',
#         how='left'
#     )
# )
# Affinity will be computed after train/val split to avoid leakage
final_df = (
    ratings_core
    .merge(user_stats, on='userId', how='left')
    .merge(movie_stats, on='movieId', how='left')
    .merge(
        meta_df[['id'] + genre_cols + [
            'log_popularity', 'log_vote_count', 'release_year',
            'runtime', 'in_collection', 'collection_size',
            'collection_id', 'vote_average', 'overview'
        ]],
        left_on='movieId', right_on='id', how='left'
    )
)
final_df = final_df.drop(columns=['id'])
# -----------------------------
# 11. Sanity checks
# -----------------------------
assert len(final_df) == len(ratings_core)
assert final_df.isna().sum().sum() > 0  # expected (cold-start / missing joins)

display(final_df.head())

,userId,movieId,rating,timestamp,rating_deviation,user_mean_rating,user_rating_std,user_rating_count,cf_mean_rating,cf_rating_std,...,genre_Western,log_popularity,log_vote_count,release_year,runtime,in_collection,collection_size,collection_id,vote_average,overview
0,1,147,4.5,1425942435,0.055556,4.444444,0.46398,9,4.320866,0.423788,...,0,2.112476,5.897154,1959.0,99.0,1,5,381,8.0,"For young Parisian boy Antoine Doinel, life is one difficult situation after another. Surrounded by inconsiderate ad..."
1,1,858,5.0,1425941523,0.555556,4.444444,0.46398,9,4.652307,0.433015,...,0,2.419027,6.447306,1993.0,105.0,0,0,0,6.5,A young boy who tries to set his dad up on a date after the death of his mother. He calls into a radio station to ta...
2,1,1246,5.0,1425941556,0.555556,4.444444,0.46398,9,4.401756,0.438170,...,0,2.541413,6.755769,2006.0,102.0,1,6,323,6.5,"When he loses a highly publicized virtual boxing match to ex-champ Rocky Balboa, reigning heavyweight titleholder, M..."
3,1,1968,4.0,1425942148,-0.444444,4.444444,0.46398,9,4.378298,0.445452,...,0,1.985896,4.897840,1997.0,109.0,0,0,0,5.8,Alex Whitman (Matthew Perry) is a designer from New York City who is sent to Las Vegas to supervise the construction...
4,1,2762,4.5,1425941300,0.055556,4.444444,0.46398,9,4.450370,0.447257,...,0,1.573787,3.761200,1937.0,83.0,0,0,0,6.8,Derrick De Marney finds himself in a 39 Steps situation when he is wrongly accused of murder. While a fugitive from ...


## 7. Fill missing values and encode IDs

The final table is cleaned, indexed, and ready for model training.

In [7]:
final_df['release_year'] = final_df['release_year'].fillna(final_df['release_year'].median()).astype(int)
final_df['runtime'] = final_df['runtime'].fillna(final_df['runtime'].median())

user_id_map = {uid: idx for idx, uid in enumerate(final_df['userId'].unique())}
movie_id_map = {mid: idx for idx, mid in enumerate(final_df['movieId'].unique())}
final_df['user_idx'] = final_df['userId'].map(user_id_map)
final_df['movie_idx'] = final_df['movieId'].map(movie_id_map)

assert final_df['user_idx'].isna().sum() == 0
assert final_df['movie_idx'].isna().sum() == 0

print('Final shape:', final_df.shape)
print('Users:', final_df['user_idx'].nunique())
print('Movies:', final_df['movie_idx'].nunique())

Final shape: (5747533, 43)
Users: 254114
Movies: 5972


## 8. Sample and split

We sample users for quicker experiments, then create a user-based train/validation split.

In [8]:
sample_user_count = min(50000, final_df['user_idx'].nunique())
sampled_users = np.random.choice(final_df['user_idx'].unique(), size=sample_user_count, replace=False)
sample_df = final_df[final_df['user_idx'].isin(sampled_users)].reset_index(drop=True)




## ENCONDING

In [9]:
#from sentence_transformers import SentenceTransformer

#encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Get unique movies in stable order
unique_movies = (
    sample_df[['movieId', 'overview']]
    .drop_duplicates(subset='movieId')
    .sort_values('movieId')
    .reset_index(drop=True)
)

# overview_embeddings = encoder.encode(
#     unique_movies['overview'].fillna('').tolist(),
#     batch_size=256,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )
#print(f"Embeddings shape: {overview_embeddings.shape}")  # (4334, 384)

#np.save("overview_embeddings.npy", overview_embeddings)
overview_embeddings = np.load("overview_embeddings.npy")

assert len(unique_movies) == overview_embeddings.shape[0], "Mismatch between movies and embeddings!"

#overview_cols = [f'overview_{i}' for i in range(overview_embeddings.shape[1])]
#overview_lookup = pd.DataFrame(overview_embeddings, columns=overview_cols)

#overview_lookup['movieId'] = unique_movies['movieId'].values
from sklearn.decomposition import PCA

# After generating overview_embeddings
pca = PCA(n_components=128, random_state=42)
overview_pca = pca.fit_transform(overview_embeddings)

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.3f}")

overview_cols = [f'overview_pca_{i}' for i in range(128)]
overview_lookup = pd.DataFrame(overview_pca, columns=overview_cols)
overview_lookup['movieId'] = unique_movies['movieId'].values

# Merge
samples_df_with_overview = sample_df.merge(overview_lookup, on='movieId', how='left')

# sanity check
print(f"Shape: {samples_df_with_overview.shape}")
print("Missing embeddings:", samples_df_with_overview[overview_cols].isnull().sum().sum())

Variance explained: 0.814
Shape: (1137713, 171)
Missing embeddings: 0


In [10]:
# Split FIRST
unique_users = samples_df_with_overview['user_idx'].unique()
np.random.shuffle(unique_users)
split_idx   = int(0.85 * len(unique_users))
train_users = set(unique_users[:split_idx])
val_users   = set(unique_users[split_idx:])

train_df = samples_df_with_overview[samples_df_with_overview['user_idx'].isin(train_users)].reset_index(drop=True)
val_df   = samples_df_with_overview[samples_df_with_overview['user_idx'].isin(val_users)].reset_index(drop=True)

# All stats from train only
user_stats = train_df.groupby('userId')['rating'].agg(
    user_mean_rating='mean', user_rating_std='std', user_rating_count='count'
).reset_index().fillna(0)
user_stats['user_rating_count'] = np.log1p(user_stats['user_rating_count'])  # Fix 7 too

movie_stats = train_df.groupby('movieId')['rating'].agg(
    cf_mean_rating='mean', cf_rating_std='std', cf_rating_count='count'
).reset_index().fillna(0)
movie_stats['log_cf_count'] = np.log1p(movie_stats['cf_rating_count'])

user_mean_map = user_stats.set_index('userId')['user_mean_rating']
train_df['rating_deviation'] = train_df['rating'] - train_df['userId'].map(user_mean_map)
val_df['rating_deviation']   = val_df['rating']   - val_df['userId'].map(user_mean_map).fillna(user_mean_map.mean())

# Drop old stats columns if already merged, re-merge clean
for df in [train_df, val_df]:
    for col in ['user_mean_rating','user_rating_std','user_rating_count','cf_mean_rating','cf_rating_std','cf_rating_count','log_cf_count']:
        if col in df.columns: df.drop(columns=col, inplace=True)

train_df = train_df.merge(user_stats, on='userId', how='left').merge(movie_stats[['movieId','cf_mean_rating','cf_rating_std','log_cf_count']], on='movieId', how='left')
val_df   = val_df.merge(user_stats, on='userId', how='left').merge(movie_stats[['movieId','cf_mean_rating','cf_rating_std','log_cf_count']], on='movieId', how='left')

In [11]:
affinity_cols = [c for c in train_df.columns if c.startswith('affinity_')]

USER_FEATS = [
    'user_mean_rating', 'user_rating_std',
    'user_rating_count', 'rating_deviation'   # Fix 4: added rating_deviation
] + affinity_cols

ITEM_FEATS = [
    'cf_mean_rating', 'cf_rating_std', 'log_cf_count',
    'log_popularity', 'log_vote_count', 'release_year',
    'runtime', 'vote_average', 'in_collection', 'collection_size'
] + genre_cols + overview_cols

# User scaler — fit on train rows (ok, user feats vary per interaction)
user_scaler = StandardScaler()
train_df[USER_FEATS] = user_scaler.fit_transform(train_df[USER_FEATS])
val_df[USER_FEATS]   = user_scaler.transform(val_df[USER_FEATS])

# Item scaler — fit on unique items only (Fix 3)
unique_items = train_df.drop_duplicates(subset='movieId')
item_scaler  = StandardScaler().fit(unique_items[ITEM_FEATS])
train_df[ITEM_FEATS] = item_scaler.transform(train_df[ITEM_FEATS])
val_df[ITEM_FEATS]   = item_scaler.transform(val_df[ITEM_FEATS])

print(f"User feat dim: {len(USER_FEATS)} | Item feat dim: {len(ITEM_FEATS)}")

User feat dim: 4 | Item feat dim: 158


In [12]:
# overview text not needed in model file, embeddings are already expanded
train_df = train_df.drop(columns=['overview'], errors='ignore')
val_df   = val_df.drop(columns=['overview'], errors='ignore')

train_df.to_parquet("train_df.parquet", index=False)
val_df.to_parquet("val_df.parquet", index=False)

### 8.1 Save samples

In [13]:

import pickle

samples_df_with_overview.to_parquet("sample_df_20000_new.parquet", index=False)

with open('user_scaler.pkl', 'wb') as f: pickle.dump(user_scaler, f)
with open('item_scaler.pkl', 'wb') as f: pickle.dump(item_scaler, f)

print("Saved parquet + scalers.")   

train_df.to_parquet("train_df_128.parquet", index=False)
val_df.to_parquet("val_df_128.parquet", index=False)


print("Saved train, val and scalers.")

Saved parquet + scalers.
Saved train, val and scalers.


## PLAYING

In [14]:
samples_df_with_overview.columns

Index(['userId', 'movieId', 'rating', 'timestamp', 'rating_deviation',
       'user_mean_rating', 'user_rating_std', 'user_rating_count',
       'cf_mean_rating', 'cf_rating_std',
       ...
       'overview_pca_118', 'overview_pca_119', 'overview_pca_120',
       'overview_pca_121', 'overview_pca_122', 'overview_pca_123',
       'overview_pca_124', 'overview_pca_125', 'overview_pca_126',
       'overview_pca_127'],
      dtype='str', length=171)